In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

from rockyclickup import wrapper, models, utils


from odin import PostgresWrapper




In [ ]:
'''initialize sessions'''

rcu = wrapper.Session()

identity_db = PostgresWrapper(
    instance_connection_name=os.getenv("IDENTITY_DB_CONNECTION_NAME"),
    db_name=os.getenv("IDENTITY_DB_NAME"),
    user=os.getenv("IDENTITY_USER"),
    password=os.getenv("IDENTITY_PASSWORD"),
    ip_type="public",
)

In [ ]:
# get all clients
all_clients = rcu.get_full_list(models.Client.list_id)

# turn it into a dataframe
client_df = utils.response_to_dataframe(all_clients)

# extract a list of rmrcodes
all_rmrcodes = list(client_df['rmr_code'])

In [ ]:
# these rmrcodes appear on more than one client:
duplicate_rmrcodes = [
    "RMRCMF",           # 2
    "RMRSYN",           # 2
    "RMRCLF",           # 2
    "RMRSTH",           # 2
    "RMRTHA",           # 2
    "RMRCES",           # 2
    None,               # 439
]   

rmrcodes = [r for r in all_rmrcodes if r not in duplicate_rmrcodes]


In [ ]:
# look for them in the identity database

# instead of making a query for every rmrcode,
# just pull the all rmrcodes once and filter after

rmrcode_identities = identity_db.query(
    """
        SELECT * FROM identity_crosswalk
        WHERE system_id = %s
    """,
    [4]
)

rmrcode_identity_lookup = {
    i.get("external_id"): i
    for i in rmrcode_identities
}




In [ ]:
rmrcode_identities[0]

In [ ]:

# for any given client, we need to find the "broker of contact"

my_rmrcode = "RMRUSAD"

def get_contacts(rmrcode: str = None):
    if rmrcode is None:
        raise ValueError("rmrcode parameter must be provided")

    # get identity entity id by rmrcode

    result = identity_db.query_one(
        """
            SELECT entity_id FROM identity_crosswalk
            WHERE external_id = %s
            AND system_id = %s
        """,
        [rmrcode, 4]
    )

    if result is None:
        raise ValueError(f"rmrcode {rmrcode} does not appear in the identity database")

    entity_id = result.get("entity_id")

    print(f"entity_id = {entity_id}")

    # get clickup client id by entity id

    result = identity_db.query_one(
        """
            SELECT * FROM identity_crosswalk
            WHERE entity_id = %s
            AND system_id = %s
        """,
        [entity_id, 1]
    )

    if result is None:
        raise ValueError(f"clickup identity for rmrcode {rmrcode} does not appear in the identity database")

    client_id = result.get("external_id")

    print(f"clickup_id = {client_id}")

    # get client task from clickup

    client_res = rcu.get_task_by_id(client_id)

    # TODO add error handling
    # what if this task id no longer exists on clickup

    client_model:models.Client = utils.response_to_model(client_res)
    
    # there are tasks in this relation field that are under the `todo` list
    # they are removed in the later list comprehension turning these into models
    contact_ids = client_model.client_contacts

    # get contacts from clickup

    contacts_res = rcu.get_tasks(contact_ids)

    contact_models:list[models.Contact] = [
        utils.response_to_model(res)
        for res in contacts_res
        if res.get("list", {}).get("id") == "901102730845" # `todo` list id
    ]

    # loop through contacts and collect brokerages

    brokerages = []
    for contact in contact_models:
        brokerages.extend(contact.brokerage)

    # get brokerages from clickup

    broker_res = rcu.get_tasks(brokerages)

    broker_models:list[models.Brokerage] = [
        utils.response_to_model(res)
        for res in broker_res
    ]


    return broker_models

contacts = get_contacts(my_rmrcode)




In [ ]:
len(contacts)

In [ ]:
random_list = rcu.get_list(901102582315)

In [ ]:
random_list